# Read-only Iceberg HadoopCatalog exploration

This notebook inspects a persistent HadoopCatalog created by the Java loader. It deliberately contains no write, maintenance, or catalog-mutation commands.

For local use, start the synthetic stack with `--runtime-dir .local-notebook` and set `ICEBERG_WAREHOUSE` as described in `notebooks/README.md`. GCS requires both `NOTEBOOK_ENABLE_GCS=true` and an explicit `gs://` warehouse.

In [2]:
from pathlib import Path
import os
import sys

# Kernels may start in the notebook directory rather than the checkout root.
configured_root = os.environ.get('GCE_HADOOP_CATALOG_REPOSITORY')
candidates = ([Path(configured_root).expanduser()] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    (path.resolve() for path in candidates if (path / 'src' / 'gce_hadoop_catalog').is_dir()),
    None,
)
if repository_root is None:
    raise RuntimeError(
        'Cannot find the checkout. Start Jupyter from the repository or set '
        'GCE_HADOOP_CATALOG_REPOSITORY=/path/to/gcp_alpaca_datalake.'
    )
sys.path.insert(0, str(repository_root / 'src'))

from gce_hadoop_catalog.spark_catalog import create_spark_session, settings_from_environment

settings = settings_from_environment(repository_root)
print({
    'catalog': settings.catalog_name,
    'warehouse': settings.warehouse,
    'namespace': settings.namespace,
    'table': settings.table,
    'gcs_enabled': settings.gcs_enabled,
})
spark = create_spark_session(settings)
spark.conf.get('spark.sql.session.timeZone')

{'catalog': 'alpaca', 'warehouse': 'file:///home/kumararpita/gcp_alpaca_datalake/.local-notebook/warehouse', 'namespace': 'alpaca', 'table': 'bars_raw', 'gcs_enabled': False}


26/08/17 12:31:38 WARN Utils: Your hostname, kumararpita-OMEN-Laptop-15-en0xxx resolves to a loopback address: 127.0.1.1; using 192.168.1.22 instead (on interface wlo1)
26/08/17 12:31:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/kumararpita/gcp_alpaca_datalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/kumararpita/.ivy2/cache
The jars for the packages stored in: /home/kumararpita/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2cec2fa-0568-45f6-afa4-d264c877fb25;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
:: resolution report :: resolve 160ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org

'UTC'

In [3]:
print({
    'catalog': settings.catalog_name,
    'warehouse': settings.warehouse,
    'namespace': settings.namespace,
    'table': settings.table,
    'gcs_enabled': settings.gcs_enabled,
})

{'catalog': 'alpaca', 'warehouse': 'file:///home/kumararpita/gcp_alpaca_datalake/.local-notebook/warehouse', 'namespace': 'alpaca', 'table': 'bars_raw', 'gcs_enabled': False}


## Inspect the configured namespace

`NOTEBOOK_NAMESPACE` and `NOTEBOOK_TABLE` override the defaults (`alpaca` and `bars_raw`) without editing this notebook. HadoopCatalog does not reliably list a completely empty warehouse root, so this notebook inspects the configured namespace directly.

In [4]:
namespace = f'{settings.catalog_name}.{settings.namespace}'
try:
    spark.sql(f'SHOW TABLES IN {namespace}').show(truncate=False)
except Exception as error:
    raise RuntimeError(
        f'Namespace {namespace!r} is absent from {settings.warehouse!r}. '
        'For persistent local data, run: '
        'uv run python scripts/run_local_stack.py --source synthetic --runtime-dir .local-notebook; '
        'then restart Jupyter with ICEBERG_WAREHOUSE="$PWD/.local-notebook/warehouse".'
    ) from error

Py4JJavaError: An error occurred while calling o42.sql.
: org.apache.iceberg.exceptions.NoSuchNamespaceException: Namespace does not exist: 
	at org.apache.iceberg.hadoop.HadoopCatalog.listNamespaces(HadoopCatalog.java:308)
	at org.apache.iceberg.catalog.SupportsNamespaces.listNamespaces(SupportsNamespaces.java:74)
	at org.apache.iceberg.spark.SparkCatalog.listNamespaces(SparkCatalog.java:433)
	at org.apache.spark.sql.execution.datasources.v2.ShowNamespacesExec.run(ShowNamespacesExec.scala:42)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.result$lzycompute(V2CommandExec.scala:43)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.result(V2CommandExec.scala:43)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.executeCollect(V2CommandExec.scala:49)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyExecuteCommands$1.$anonfun$applyOrElse$1(QueryExecution.scala:107)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNe

In [5]:
table = settings.table_identifier
spark.sql(f'DESCRIBE TABLE {table}').show(truncate=False)
spark.sql(f'SELECT committed_at, snapshot_id, operation, summary FROM {table}.snapshots ORDER BY committed_at DESC').show(truncate=False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `alpaca`.`alpaca`.`bars_raw` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 15;
'DescribeRelation false, [col_name#2, data_type#3, comment#4]
+- 'UnresolvedTableOrView [alpaca, alpaca, bars_raw], DESCRIBE TABLE, true


## Bounded bar query

The table retains Alpaca's case-sensitive wire contract: `T` is the event type and `t` is the UTC RFC 3339 timestamp. Spark is therefore configured as case-sensitive. Override `NOTEBOOK_START_UTC`, `NOTEBOOK_END_UTC`, and `NOTEBOOK_LIMIT` for another bounded inspection.

In [ ]:
start_utc = os.environ.get('NOTEBOOK_START_UTC', '2026-01-01T00:00:00Z')
end_utc = os.environ.get('NOTEBOOK_END_UTC', '2026-01-03T00:00:00Z')
limit = int(os.environ.get('NOTEBOOK_LIMIT', '100'))

bars = spark.sql(
    f"""
    SELECT S AS symbol, t, o, h, l, c, v, n, vw, ingested_at, payload_hash
    FROM {table}
    WHERE t >= '{start_utc}' AND t < '{end_utc}'
    ORDER BY t, S
    LIMIT {limit}
    """
)
bars.show(truncate=False)

In [ ]:
spark.sql(
    f"EXPLAIN FORMATTED SELECT * FROM {table} WHERE t >= '{start_utc}' AND t < '{end_utc}'"
).show(truncate=False)